In [1]:
import pandas as pd

In [14]:
df = pd.read_csv("heart.csv")
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [15]:
x = df.drop(columns=["target"])
y = df["target"]

### Important:
- For Training data use **.fit_transform**, but for testing data only use **.transform** to prevent data leakage.

In [16]:
from sklearn.model_selection import train_test_split

In [17]:
x_train, x_test, y_train, y_test = train_test_split(x, y, random_state=42, test_size=0.2)

# Stamdardization

In [3]:
from sklearn.preprocessing import StandardScaler

In [8]:
Scaler = StandardScaler()
x_train = Scaler.fit_transform(x_train)
x_test =  Scaler.fit_transform(x_test)

In [9]:
pd.DataFrame(x_train).head()

,0,1,2,3,4,5,6,7,8,9,10,11,12
0,-1.356798,0.722504,0.008099,-0.616856,0.914034,-0.383301,0.843133,0.532781,-0.676632,-0.920864,0.953905,-0.689701,-0.509048
1,0.385086,0.722504,-0.971891,1.169491,0.439527,-0.383301,-1.046109,-1.753582,1.477907,-0.193787,0.953905,-0.689701,1.178480
2,-0.921327,0.722504,0.988089,1.169491,-0.300704,-0.383301,0.843133,-0.139679,-0.676632,2.350982,-0.694988,-0.689701,-0.509048
3,0.058483,-1.384075,0.008099,0.276318,0.059921,-0.383301,-1.046109,0.487950,-0.676632,0.351521,-0.694988,-0.689701,-0.509048
4,0.602822,0.722504,-0.971891,-0.795490,-0.319684,2.608918,0.843133,0.443119,1.477907,0.351521,0.953905,1.333421,1.178480


# Normalization

In [21]:
from sklearn.preprocessing import MinMaxScaler

In [22]:
ScaleMinMax = MinMaxScaler(feature_range=(0,1))
x_train = ScaleMinMax.fit_transform(x_train)
x_test = ScaleMinMax.transform(x_test)

In [23]:
pd.DataFrame(x_train).head()

,0,1,2,3,4,5,6,7,8,9,10,11,12
0,0.270833,1.0,0.333333,0.265306,0.378753,0.0,0.5,0.649123,0.0,0.000000,1.0,0.0,0.666667
1,0.604167,1.0,0.000000,0.571429,0.321016,0.0,0.0,0.201754,1.0,0.142857,1.0,0.0,1.000000
2,0.354167,1.0,0.666667,0.571429,0.230947,0.0,0.5,0.517544,0.0,0.642857,0.5,0.0,0.666667
3,0.541667,0.0,0.333333,0.418367,0.274827,0.0,0.0,0.640351,0.0,0.250000,0.5,0.0,0.666667
4,0.645833,1.0,0.000000,0.234694,0.228637,1.0,0.5,0.631579,1.0,0.250000,1.0,0.5,1.000000


Excellent question — this is a common point of confusion. Both **standardization** and **normalization** are **feature scaling** techniques, but they work differently and are used in different situations.

Let me explain with simple intuition first, then the rules.

---

## The Core Difference (Simple Version)

| Technique | What it does | Output range |
|-----------|--------------|---------------|
| **Normalization** (Min-Max Scaling) | Shifts values to a fixed range, usually **[0, 1]** | [0, 1] (or [-1, 1]) |
| **Standardization** (Z-score Scaling) | Centers data around **mean=0**, with **standard deviation=1** | No fixed range (typically -3 to +3) |

---

## 1. Normalization (Min-Max Scaling)

**Formula:**  
X_norm = (X - X_min) / (X_max - X_min)

**What it does:**  
Rescales data so minimum becomes 0, maximum becomes 1, and everything else falls in between.

**Example:**  
Ages [20, 40, 60, 80] → normalized → [0, 0.33, 0.67, 1]

**Use when:**
- You **know the data has bounded limits** (e.g., pixel values 0-255, percentages 0-100)
- Your algorithm **expects data in a specific range** (usually [0,1] or [-1,1])
- You have **no outliers** (outliers will compress the rest of the data)

**Algorithms that need normalization:**
- **Neural networks** (especially with sigmoid/tanh activation)
- **K-Nearest Neighbors (KNN)**
- **K-Means clustering**
- **Principal Component Analysis (PCA)** — *actually standardization is better here, but normalization works*

---

## 2. Standardization (Z-score Scaling)

**Formula:**  
X_std = (X - mean) / standard_deviation

**What it does:**  
Centers data at 0, with units in "number of standard deviations from the mean."

**Example:**  
Ages [20, 40, 60, 80]  
Mean=50, Std≈22.36  
Standardized → [-1.34, -0.45, 0.45, 1.34]

**Use when:**
- Your data has **outliers** (standardization is robust to them)
- Your algorithm **assumes normally distributed data**
- You don't know the min/max bounds of your data
- You're using algorithms that **assume zero-centered data**

**Algorithms that need standardization:**
- **Linear Regression** (coefficients interpretable)
- **Logistic Regression**
- **Support Vector Machines (SVM)**
- **Principal Component Analysis (PCA)** — *preferred over normalization*
- **Lasso/Ridge regression**
- **Neural networks with ReLU** (often works better with standardization)
- **Any algorithm using distance** (like KNN) — *standardization handles outliers better*

---

## Quick Decision Flowchart

```
Do you have outliers?
    ├─ YES → Use STANDARDIZATION
    └─ NO  → Continue

Does your algorithm assume normal distribution?
    ├─ YES → Use STANDARDIZATION
    └─ NO  → Continue

Do you need values in a fixed range (e.g., [0,1] for images)?
    ├─ YES → Use NORMALIZATION
    └─ NO  → Either works, but standardization is safer

Do you know exact min/max bounds of your data?
    ├─ YES (e.g., pixel values 0-255) → Use NORMALIZATION
    └─ NO → Use STANDARDIZATION
```

---

## Side-by-Side Example

**Original data:** [1, 2, 3, 4, 100] (note the outlier 100)

| Method | Result | Problem? |
|--------|--------|----------|
| **Normalization** | [0, 0.01, 0.02, 0.03, 1] | Normal values (1-4) get crushed near 0 because 100 dominates |
| **Standardization** | [-0.68, -0.67, -0.66, -0.65, 2.66] | Normal values (1-4) stay close together and near 0; outlier is far away (good) |

**Winner with outliers:** Standardization

---

## When It REALLY Matters

| Algorithm | Prefers | Why |
|-----------|---------|-----|
| **KNN** | Standardization (or normalization) | Distance-based; scale must not dominate |
| **Neural Networks** | Depends on activation: Normalization for sigmoid/tanh, Standardization for ReLU | Keeps activations in good range |
| **Linear Regression** | Standardization | Coefficients become comparable (importance) |
| **Decision Trees / Random Forest** | **NEITHER** | Tree-based algorithms are scale-invariant |
| **PCA** | Standardization | Finds directions of maximum variance fairly |
| **Clustering (K-Means)** | Standardization (safer) | Distance-based; outliers hurt less |

---

## Simple Rule of Thumb (Beginners)

**When in doubt → use STANDARDIZATION.**  
It works well for most algorithms and handles outliers gracefully.

**Only use NORMALIZATION when:**
- You're working with **images** (pixel values 0-255)
- You **know** your data has fixed bounds (e.g., percentages 0-100)
- You're using a **neural network with sigmoid/tanh** (they expect [0,1] or [-1,1])

---

## Remember This

- **Normalization** = squishing data into a box ([0,1])
- **Standardization** = centering data around zero (mean=0, std=1)

**Analogy:**  
- Normalization is like resizing all photos to 100×100 pixels  
- Standardization is like converting temperatures to Celsius (zero-centered, consistent units)

Both help algorithms that are sensitive to scale — trees don't care, distances and gradients do!